In [19]:
from dotenv import load_dotenv

load_dotenv()

True

In [20]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [21]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str
    
agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [22]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response.")],
        "email": "Hi Mitchell, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [23]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Subject: '
                                                                          'Re: '
                                                                          'Meeting '
                                                                          'tomorrow\n'
                                                                          '\n'
                                                                          'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
      

In [24]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Subject: Re: Meeting tomorrow\n\nHi John,\n\nThanks for the heads-up. No problem—let's reschedule. I'm available tomorrow between 9–11 AM and 1–4 PM. Do any of these times work for you? If not, please suggest a time that suits you and I'll adjust.\n\nBest regards,\nMitchell"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Subject: Re: Meeting tomorrow\\n\\nHi John,\\n\\nThanks for the heads-up. No problem—let\'s reschedule. I\'m available tomorrow between 9–11 AM and 1–4 PM. Do any of these times work for you? If not, please suggest a time that suits you and I\'ll adjust.\\n\\nBest regards,\\nMitchell"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='a50ca8e51e5be35f65da25a6f17004ab')]


In [25]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Subject: Re: Meeting tomorrow

Hi John,

Thanks for the heads-up. No problem—let's reschedule. I'm available tomorrow between 9–11 AM and 1–4 PM. Do any of these times work for you? If not, please suggest a time that suits you and I'll adjust.

Best regards,
Mitchell


## Accept

In [9]:
from langgraph.types import Command
response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Mitchell, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response.', additional_kwargs={}, response_metadata={}, id='2b1092f4-82db-4889-8350-7e24dd40a0ce'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 531, 'prompt_tokens': 157, 'total_tokens': 688, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9ZobKoqRNn0Ocfqay5P2emNCunvH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd2fe-a216-77f3-917a-089055bc6e3b-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'cal

## Reject

In [15]:
reponse = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Mitchell."
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all—thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'I’m '
              

In [16]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem at all—thanks for letting me know. I’m available to reschedule for tomorrow. Do any of these times work for you?

- 10:00 AM
- 1:00 PM
- 4:00 PM

If none of these fit, please tell me a time that does, and I’ll adjust.

Best regards,
Mitchell


## Edit

In [26]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'email': "Hi Mitchell, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response.', additional_kwargs={}, response_metadata={}, id='8a563841-4fb9-408a-9370-7af555fb7945'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 531, 'prompt_tokens': 157, 'total_tokens': 688, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9a58knfwjD2eB3AePoSQLRJuGj8d', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd30e-4bc7-7043-8923-655bde3ea645-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'cal